# Which Content Should an Editor Review First?

**FlyRank ML Internship — ML-11 Capstone**  
**Author:** Abdul Raheem  

[Deployed-paper source](../../docs/index.html) · [ML-10 action playbook](w07_action_playbook.ipynb)

## Abstract

This study asks how an editorial team can prioritize a limited content-refresh queue without treating every observed decline as equally urgent. I used FlyRank's anonymized 30,000-row teaching slice and excluded identifiers, labels, trend fields, and recent-window fields that would leak the outcome. A histogram gradient boosting model was trained with client-grouped validation, so seven unseen client groups were held out for evaluation. On that held-out slice, the cleaned model reached Precision@50 of 0.84, while the earlier transparent rule baseline reached 0.32 on the same grouped-split setup. The output is a ranked, human-reviewed triage aid for deciding what to inspect first; it does not predict Google's algorithm or establish that refreshing content causes recovery.

## 1. Question

**Research question:** Given limited editorial capacity, which content items should a human reviewer inspect first for possible refresh work?

The unit of analysis is a content item. The system produces a ranked review queue and an action-oriented reason code. It supports an SEO strategist or content lead deciding where to spend review time; it does not make publishing, deletion, or rewrite decisions. A wrong high-priority call costs scarce editorial time, while a missed high-value declining item can delay a useful review.

In [1]:
# Evidence is documented in ML-08 through ML-10.
# This capstone is a public-safe synthesis; no raw rows are displayed here.
print('Decision supported: human-reviewed content-refresh prioritization.')

Decision supported: human-reviewed content-refresh prioritization.


## 2. Data

This capstone uses the repository's anonymized starter dataset: 30,000 content items across 32 pseudonymized client groups, with observed search, engagement, freshness, and content metadata. It does **not** claim a full 79M-row warehouse model: no local credential or rerunnable warehouse environment was available during final assembly.

No client names, domains, URLs, titles, keywords, raw queries, or credentials appear in this notebook or the paper. `client_id` is used only to create grouped splits, and `content_id` is excluded from features. The target is derived from observed trend direction, so `trend_direction`, `trend_pct`, and last/previous 30-day performance fields are excluded from the cleaned model to avoid label-period leakage.

In [2]:
from pathlib import Path
assert Path('../../data/raw/content_refresh_anonymized.csv').exists() or Path('data/raw/content_refresh_anonymized.csv').exists()
print('Anonymized starter dataset location verified; no rows displayed.')

Anonymized starter dataset location verified; no rows displayed.


## 3. Methodology

The cleaned model uses histogram gradient boosting with `GroupShuffleSplit(test_size=0.20, random_state=42)`: 25 client groups train the model and 7 unseen client groups form the held-out test slice (6,163 items). Features are non-identifying and non-label-derived signals such as 90-day visibility, position, content age, staleness, and carefully encoded metadata.

The transparent baseline is a rule-based prioritization score. Both approaches are assessed on the grouped holdout because the practical workflow only has capacity to inspect the top of the queue. The model probability is combined with a traffic-value proxy and diagnostic reason codes, but these are recommendations for human review—not autonomous actions.

In [3]:
LEAKAGE_EXCLUSIONS = ['is_declining_label', 'trend_direction', 'trend_pct', 'content_id', 'impressions_last_30d', 'impressions_prev_30d', 'clicks_last_30d', 'clicks_prev_30d', 'sessions_last_30d', 'sessions_prev_30d']
print('Leakage exclusions documented:', len(LEAKAGE_EXCLUSIONS))

Leakage exclusions documented: 10


## 4. Results (vs baseline)

| Approach | Evaluation frame | Precision@50 | Interpretation |
|---|---:|---:|---|
| Cleaned histogram gradient boosting model | 7 unseen client groups | 0.84 | 42 of the top 50 candidates were declining under the proxy label. |
| Transparent rule baseline | Same grouped-split setup | 0.32 | 16 of the top 50 candidates were declining under the proxy label. |

The held-out decline base rate is 51.1%. The observed result is a ranking improvement on this client-held-out slice, not a causal estimate of refresh impact. The supporting action playbook maps candidates to title/snippet review, readability review, freshness review, depth audit, standard review, or monitor-only.

In [4]:
import json
paths = [Path('../outputs/playbook_summary.json'), Path('work/outputs/playbook_summary.json')]
summary_path = next((p for p in paths if p.exists()), None)
if summary_path:
    metrics = json.loads(summary_path.read_text(encoding='utf-8'))['model_metrics']
    print('Saved ML-10 evidence receipt:', metrics)
else:
    print('Run ML-10 to regenerate work/outputs/playbook_summary.json before final submission.')

Saved ML-10 evidence receipt: {'precision_at_10': 0.9, 'precision_at_20': 0.9, 'precision_at_50': 0.84, 'roc_auc': 0.6179, 'pr_auc': 0.6166}


## 5. Limitations

The label is a contemporaneous decline proxy rather than a future causal outcome. A single grouped split does not replace a time-forward validation design, and seasonality, search-engine changes, and changing client mix can alter the feature–outcome relationship.

The capstone is reproducible from the committed anonymized teaching slice, but it is not a 79M-row full-warehouse study. It should be read as directional decision support. No recommendation demonstrates that a content refresh will restore ranking, traffic, or conversions.

In [5]:
print('Boundary: observed ranking evidence; no causal refresh-impact claim.')

Boundary: observed ranking evidence; no causal refresh-impact claim.


## 6. Ranked recommendations

1. Start with high-value, high-risk candidates, then confirm search intent and the live SERP before assigning work.
2. Match intervention to the diagnostic: title/snippet review for CTR opportunity, readability review for engagement friction, and freshness/depth review only after manual evidence supports it.
3. Deprioritize low-demand long-tail items; monitor-only is an explicit capacity decision.
4. Never automatically rewrite, publish, delete URLs, bulk-change metadata, or operate during a migration or technical incident.
5. Track realized editor-reviewed outcomes and retrain only after validating drift and feedback effects.

In [6]:
print('Ranked recommendations are human-review prompts, not automated commands.')

Ranked recommendations are human-review prompts, not automated commands.


## 7. Artifacts the paper embeds

The published paper embeds three public-safe ML-10 figures: the portfolio action mix, the traffic-value versus decline-risk quadrant, and monitoring/retrain triggers. It also presents the same-split Precision@50 comparison and the held-out base rate.

The paper is authored at `docs/index.html` with relative assets in `docs/assets/`. Before publishing, rerun this notebook and ML-08 through ML-10 in a configured environment, then re-check that these figures and metrics match.

In [7]:
artifact_names = ['index.html', 'assets/archetype_action_matrix.png', 'assets/value_vs_urgency_quadrant.png', 'assets/monitoring_retrain_triggers.png']
for name in artifact_names:
    candidates = [Path('docs') / name, Path('../../docs') / name]
    resolved = next((path for path in candidates if path.exists()), None)
    print(name, resolved is not None)

index.html True
assets/archetype_action_matrix.png True
assets/value_vs_urgency_quadrant.png True
assets/monitoring_retrain_triggers.png True


## 8. Reproducibility and acknowledgments

The evidence trail is in `w05_model.ipynb`, `w06_validation_audit.ipynb`, `w07_action_playbook.ipynb`, and `work/outputs/playbook_summary.json`. Install `requirements.txt`, run the reference pipeline with `python scripts/run_all.py`, then run the evidence notebooks and this notebook top-to-bottom. The grouped split uses seed 42.

**Acknowledgments & data credit:** Built on the [FlyRank ML Internship dataset](https://flyrank.ai). No raw client data or private queries are included.

## ML-12 closing materials

**Five-minute demo outline:** (1) editorial triage problem; (2) public-safe dataset and leakage exclusions; (3) client-grouped comparison; (4) action-playbook figures and human checks; (5) limitations, monitoring, and reproducibility.

**Social-post cut:** I built a client-grouped content-refresh prioritization workflow on FlyRank's anonymized teaching dataset. The cleaned model improved held-out top-50 precision from 0.32 for a transparent rule to 0.84, but the output stays firmly human-reviewed decision support. The paper explains the leakage safeguards, action playbook, and limits.

**Employer-facing summary:** I translated a machine-learning ranking problem into a deployable, public-safe research artifact. The project uses grouped validation, explicit leakage exclusions, transparent baseline comparison, and operational safeguards. I designed the output around a real decision: where an editor should spend review time first.

## Final self-check

- [x] Every required paper section is drafted in the notebook and `docs/index.html`.
- [ ] Notebook rerun in the current environment — blocked: no local Python installation is available.
- [x] No client names, URLs, private queries, or credentials are present.
- [x] Claims use observed, measured, directional, and decision-support language.
- [x] Abstract and FlyRank data credit are included.
- [ ] Paper deployed and URL written to `submission/paper_url.txt` — blocked until a commit/push is authorized.
